# SOD1 Carrier Status from UK Biobank WGS + ClinVar

This notebook:
1. Installs PLINK2 and downloads ClinVar
2. Extracts SOD1 variants from ClinVar
3. Runs 3 analysis tiers (pathogenic-only, expanded, all variants) using a **single reusable pipeline**

Each tier produces a `<prefix>_carrier_status_by_id.txt` file with IID and Carrier/Non-carrier status.

## Step 1: Setup — Install PLINK2 & Download ClinVar

In [ ]:
# Install plink2
wget -q https://s3.amazonaws.com/plink2-assets/plink2_linux_avx2_20260110.zip
unzip -o plink2_linux_avx2_20260110.zip
./plink2 --version

In [ ]:
# Download ClinVar VCF (GRCh38)
wget -q ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz
wget -q ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi

In [ ]:
# Extract all SOD1 variants from ClinVar
zgrep "GENEINFO=SOD1:" clinvar.vcf.gz > sod1_clinvar_by_name.vcf
echo "Total SOD1 variants in ClinVar: $(wc -l < sod1_clinvar_by_name.vcf)"

## Step 2: Define the reusable pipeline

The function `run_carrier_pipeline` takes a label and a filtered VCF, then:
1. Builds PLINK-format variant IDs
2. Extracts matching variants from UKB WGS data
3. Exports additive dosage (`.raw`)
4. Counts carriers and writes a carrier-status file

In [ ]:
PFILE="/mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, PLINK format [500k release]/ukb24308_c21_b0_v1"

run_carrier_pipeline() {
    local LABEL=$1   # e.g. "pathogenic", "expanded", "all"
    local VCF=$2     # filtered VCF input

    echo "===== Running pipeline: ${LABEL} ====="
    echo "Input VCF: $(wc -l < ${VCF}) variants"

    # 1. Build PLINK variant IDs
    awk '!/^#/ {print "DRAGEN:chr"$1":"$2":"$4":"$5}' "${VCF}" > sod1_${LABEL}_ids.txt

    # 2. Extract from UKB WGS
    ./plink2 \
        --pfile "${PFILE}" \
        --extract sod1_${LABEL}_ids.txt \
        --no-psam-pheno \
        --make-bed \
        --out UKB_sod1_${LABEL}

    # 3. Build allele-count file & export additive dosage
    awk -F: '{print $0"\t"$5}' sod1_${LABEL}_ids.txt > sod1_${LABEL}_alleles.txt

    ./plink2 \
        --bfile UKB_sod1_${LABEL} \
        --export A \
        --export-allele sod1_${LABEL}_alleles.txt \
        --out sod1_${LABEL}

    # 4. Count carriers & write carrier-status file
    local N_CARRIERS=$(awk 'NR>1 {for(i=7;i<=NF;i++) if($i>0){c[$2]=1;next}} END{print length(c)}' sod1_${LABEL}.raw)
    echo "Carriers found: ${N_CARRIERS}"

    awk 'BEGIN{OFS="\t"} NR==1{print "IID","Carrier_Status"; next} {
        s="Non-carrier"; for(i=7;i<=NF;i++) if($i>0){s="Carrier"; break}
        print $2, s
    }' sod1_${LABEL}.raw > sod1_${LABEL}_carrier_status_by_id.txt

    echo "Output: sod1_${LABEL}_carrier_status_by_id.txt"
    echo ""
}

## Step 3: Run the 3 analysis tiers

In [ ]:
# Tier 1: Pathogenic / Likely pathogenic only
grep -E "CLNSIG=Pathogenic|CLNSIG=Likely_pathogenic" sod1_clinvar_by_name.vcf > sod1_pathogenic.vcf
run_carrier_pipeline "pathogenic" "sod1_pathogenic.vcf"

In [ ]:
# Tier 2: Pathogenic + Likely pathogenic + Uncertain + Conflicting
grep -E "CLNSIG=Pathogenic|CLNSIG=Likely_pathogenic|CLNSIG=Uncertain|CLNSIG=Conflicting" sod1_clinvar_by_name.vcf > sod1_expanded.vcf
run_carrier_pipeline "expanded" "sod1_expanded.vcf"

In [ ]:
# Tier 3: All SOD1 variants (no CLNSIG filter)
run_carrier_pipeline "all" "sod1_clinvar_by_name.vcf"

## Step 4: Quick check

In [ ]:
for LABEL in pathogenic expanded all; do
    echo "--- ${LABEL} ---"
    head -5 sod1_${LABEL}_carrier_status_by_id.txt
    echo ""
done